# 03 Model dan Pelatihan (Peran C)
Notebook ini **memuat hasil yang sudah tersimpan** dari skrip, karena validasi silang bersarang memakan waktu.
Hanya `train.csv` yang dipakai; `test.csv` tidak disentuh.

Cara membuat ulang hasilnya (jalankan di terminal, dari folder utama repo):
1. `python scripts/run_cv.py` menghasilkan `results/cv_results.csv`
2. `python scripts/train_final.py` menghasilkan `results/models/*.joblib` dan `results/best_params.json`
3. `python scripts/analisis_c.py` menghasilkan ringkasan, grafik, dan fitur penting

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from IPython.display import Image, display

# notebook boleh dijalankan dari folder utama repo atau dari folder notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RES, SLIDES = ROOT / 'results', ROOT / 'slides'
pd.options.display.float_format = '{:.3f}'.format

## 1. Hasil validasi silang bersarang (per fold -> ringkasan)

In [ ]:
cv = pd.read_csv(RES / 'cv_results.csv')
print('Kolom:', cv.columns.tolist())
print('Jumlah baris:', len(cv), '| fold luar per model:', cv.groupby(['skenario', 'model']).size().unique())
metrik = ['f1', 'recall', 'precision', 'roc_auc']
ringkas = cv.groupby(['skenario', 'model'])[metrik].agg(['mean', 'std'])
ringkas

Catatan membaca tabel: Baseline selalu menebak kelas mayoritas (lulus), jadi F1 dan recall kelas *tidak lulus* = 0 dan ROC-AUC = 0,5 (tebak acak).
Selisih antarmodel yang lebih kecil daripada simpangan baku belum bisa disebut perbedaan nyata; pengujiannya dikerjakan D.

In [ ]:
# F1 rata-rata, dipivot supaya dua skenario mudah dibandingkan
ringkas['f1']['mean'].unstack(0).round(3)

## 2. Hiperparameter terbaik

In [ ]:
bp = json.load(open(RES / 'best_params.json'))
pd.DataFrame({k: str(v) for k, v in bp.items()}, index=['hiperparameter terbaik']).T

## 3. Pentingnya fitur (lima teratas)

In [ ]:
fi = pd.read_csv(RES / 'feature_importance_top5.csv')
fi.pivot_table(index=['skenario', 'model', 'peringkat'], values=['fitur', 'importance'], aggfunc='first')

## 4. Grafik

In [ ]:
display(Image(filename=str(SLIDES / 'grafik_f1_cv.png')))
display(Image(filename=str(SLIDES / 'grafik_pohon.png')))

## 5. Cek sebelum serah terima ke D

In [ ]:
modelnya = sorted((RES / 'models').glob('*.joblib'))
for f in modelnya:
    joblib.load(f)                      # harus bisa dimuat ulang
n_fold = cv.groupby(['skenario', 'model']).size()
assert n_fold.nunique() == 1,          'jumlah fold luar harus sama untuk semua model'
assert set(cv['skenario']) == {'dengan_G', 'tanpa_G'}, 'kedua skenario harus ada'
assert 'Baseline' in set(cv['model']), 'baseline harus ikut'
assert cv[metrik].notna().all().all(),  'tidak boleh ada nilai kosong'
assert len(modelnya) == 12,             'harus ada 6 model x 2 skenario'
print('OK:', len(modelnya), 'model .joblib terbaca;', n_fold.iloc[0], 'fold luar per model;',
      'test.csv tidak dimuat di notebook ini.')